In [21]:
import requests

import numpy as np
import pandas as pd
import plotly.express as px
import folium
import shutil
import glob
import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

import statsmodels.api as sm

In [3]:
citibike_df = pd.read_csv("../data/JC/JC2025_Enriched.csv")

citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes,date,month,month_name,day_of_week,hour,season
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member,6.192900,2025-01-16,2025-01,January,Thursday,17,Winter
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member,11.461350,2025-01-31,2025-01,January,Friday,6,Winter
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,21.377617,2025-01-09,2025-01,January,Thursday,16,Winter
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,22.934333,2025-01-21,2025-01,January,Tuesday,16,Winter
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,25.822100,2025-01-30,2025-01,January,Thursday,16,Winter


In [4]:
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(number_of_rides=("ride_id", "count"))
)

monthly_rides

,month,number_of_rides
0,2024-12,4
1,2025-01,101180
2,2025-02,180998
3,2025-03,235357
4,2025-04,244599
5,2025-05,279606
6,2025-06,290208
7,2025-07,322122
8,2025-08,324003
9,2025-09,346740


In [5]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides"
)

fig.show()

In [6]:
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(number_of_rides=("ride_id", "count"))
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,425768
1,Spring,759562
2,Summer,936333
0,Autumn,886197


In [10]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

In [7]:
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(number_of_departures=("ride_id", "count"))
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations

,start_station_name,number_of_departures
52,Grove St PATH,135136
58,Hoboken Terminal - Hudson St & Hudson Pl,77422
53,Hamilton Park,66940
95,River St & Newark St,64149
86,Newport PATH,62825
18,Bergen Ave & Sip Ave,61595
44,Exchange Pl,60076
0,11 St & Washington St,58413
94,River St & 1 St,57591
87,Newport Pkwy,56353


In [11]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

In [12]:
top_end_stations = (
    citibike_df
    .dropna(subset=["end_station_name"])
    .groupby("end_station_name", as_index=False)
    .agg(number_of_arrivals=("ride_id", "count"))
    .sort_values("number_of_arrivals", ascending=False)
    .head(10)
)

top_end_stations

,end_station_name,number_of_arrivals
232,Grove St PATH,143777
241,Hoboken Terminal - Hudson St & Hudson Pl,79811
233,Hamilton Park,67387
347,River St & Newark St,66339
317,Newport PATH,63112
73,Bergen Ave & Sip Ave,61646
207,Exchange Pl,60517
7,11 St & Washington St,58516
318,Newport Pkwy,56273
346,River St & 1 St,55590


In [13]:
fig = px.bar(
    top_end_stations.sort_values("number_of_arrivals"),
    x="number_of_arrivals",
    y="end_station_name",
    orientation="h",
    title="Top 10 End Stations by Number of Arrivals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)

fig.show()

In [14]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(number_of_rides=("ride_id", "count"))
)

daily_rides["date"] = pd.to_datetime(daily_rides["date"])

daily_rides.head()

,date,number_of_rides
0,2024-12-31,4
1,2025-01-01,2358
2,2025-01-02,3420
3,2025-01-03,3540
4,2025-01-04,2674


In [15]:
weather_daily = pd.read_csv("../data/JC/jersey_weather_2025.csv")

weather_daily["date"] = pd.to_datetime(weather_daily["date"])

weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,11.0,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [16]:
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,2358,11.0,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,3420,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,3540,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,2674,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


In [17]:
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     1
temperature_2m_min     1
temperature_2m_mean    1
precipitation_sum      1
rain_sum               1
snowfall_sum           1
wind_speed_10m_max     1
dtype: int64

In [22]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

In [23]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

In [24]:
day_rides = (
    citibike_df
    .groupby("day_of_week", as_index=False)
    .agg(number_of_rides=("ride_id", "count"))
)

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

day_rides["day_of_week"] = pd.Categorical(
    day_rides["day_of_week"],
    categories=day_order,
    ordered=True
)

day_rides = day_rides.sort_values("day_of_week")

day_rides

,day_of_week,number_of_rides
1,Monday,406626
5,Tuesday,448392
6,Wednesday,445903
4,Thursday,452424
0,Friday,472505
2,Saturday,423380
3,Sunday,358630


In [25]:
fig = px.bar(
    day_rides,
    x="day_of_week",
    y="number_of_rides",
    title="Number of Rides by Day of Week",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Day of Week",
    yaxis_title="Number of Rides"
)

fig.show()

In [26]:
hourly_rides = (
    citibike_df
    .groupby("hour", as_index=False)
    .agg(number_of_rides=("ride_id", "count"))
)

hourly_rides

,hour,number_of_rides
0,0,33711
1,1,19951
2,2,12749
3,3,7369
4,4,11538
5,5,30951
6,6,84098
7,7,163931
8,8,213459
9,9,138097


In [28]:
fig = px.line(
    hourly_rides,
    x="hour",
    y="number_of_rides",
    title="Number of Rides by Hour"
)

fig.update_layout(
    xaxis_title="Hour",
    yaxis_title="Number of Rides"
)

fig.show()